# BTL02 T0401 ? RQ3 Data Augmentation Ablation (Colab GPU)

Ch?y 8 code cells theo th? t?. Runner so s?nh EXP-09 none/basic v? EXP-10 advanced tr?n c?ng EfficientNet-B0 pretrained 128px recipe. K?t qu? ch? ???c xem l? th?c nghi?m sau khi run MLflow ho?n t?t; kh?ng nh?ng s? li?u m?u. C?u h?nh `GITHUB_TOKEN` v? `KAGGLE_API_TOKEN` trong Colab Secrets khi c?n. T?p PublicTest/PrivateTest lu?n d?ng eval transform deterministic, kh?ng augmentation.

In [ ]:
import os, platform, shutil, subprocess, sys, tempfile, threading, time
from pathlib import Path

BRANCH = "exp/BTL02_T0401_ablation_augmentation"
REPO_URL = "https://github.com/Duy-DS/Deep_Learning_Group.git"
REPO_DIR = Path("/content/Deep_Learning_Group")
PROJECT_DIR = REPO_DIR / "BTL02"
DRIVE_ROOT = Path("/content/drive/MyDrive/BTL02/T0401_RQ3")
DRIVE_MLRUNS = DRIVE_ROOT / "mlruns"
DRIVE_CHECKPOINTS = DRIVE_ROOT / "checkpoints"
DRIVE_LOGS = DRIVE_ROOT / "logs"
DRIVE_CSV = DRIVE_ROOT / "t0401_ablation_augmentation_24col.csv"
LOCAL_MLRUNS = Path("/content/mlruns_t0401")
LOCAL_CHECKPOINTS = Path("/content/checkpoints_t0401")
CSV_NAME = "t0401_ablation_augmentation_24col.csv"

print("Python:", platform.python_version())
import torch
if not torch.cuda.is_available():
    raise RuntimeError("T0401 training c?n Colab GPU. Ch?n Runtime > Change runtime type > GPU r?i ch?y l?i.")
print("PyTorch:", torch.__version__)
for index in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(index)
    print(f"GPU {index}: {props.name} | {props.total_memory / 1024**3:.1f} GiB")
from google.colab import drive
drive.mount("/content/drive")
for folder in (DRIVE_ROOT, DRIVE_MLRUNS, DRIVE_CHECKPOINTS, DRIVE_LOGS, LOCAL_MLRUNS, LOCAL_CHECKPOINTS):
    folder.mkdir(parents=True, exist_ok=True)


In [ ]:
def sync_tree_new_files(source, destination, stable_age_seconds=0):
    source, destination = Path(source), Path(destination)
    if not source.exists():
        return
    destination.mkdir(parents=True, exist_ok=True)
    for current_root, directories, filenames in os.walk(source):
        current = Path(current_root)
        target_root = destination / current.relative_to(source)
        target_root.mkdir(parents=True, exist_ok=True)
        directories[:] = [name for name in directories if name != ".trash"]
        for filename in filenames:
            item = current / filename
            if item.is_symlink() or filename.endswith((".tmp", ".part", ".sync-tmp")):
                continue
            stat = item.stat()
            if stable_age_seconds and time.time() - stat.st_mtime < stable_age_seconds:
                continue
            target = target_root / filename
            if target.exists() and target.stat().st_size == stat.st_size and target.stat().st_mtime_ns >= stat.st_mtime_ns:
                continue
            temporary = target.with_name(target.name + ".sync-tmp")
            shutil.copy2(item, temporary)
            copied = item.stat()
            if copied.st_size != stat.st_size or copied.st_mtime_ns != stat.st_mtime_ns or temporary.stat().st_size != stat.st_size:
                temporary.unlink(missing_ok=True)
                continue
            os.replace(temporary, target)

# Merge Drive backup into local SSD; never delete the local or Drive copy.
sync_tree_new_files(DRIVE_MLRUNS, LOCAL_MLRUNS)
sync_tree_new_files(DRIVE_CHECKPOINTS, LOCAL_CHECKPOINTS)
(LOCAL_MLRUNS / ".trash").mkdir(parents=True, exist_ok=True)
os.environ["MLFLOW_TRACKING_URI"] = LOCAL_MLRUNS.as_uri()
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
print("Local-first MLflow:", os.environ["MLFLOW_TRACKING_URI"])


In [ ]:
from google.colab import userdata

def clone_or_update_branch():
    token = None
    try:
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        pass
    git_env = os.environ.copy()
    askpass_path = None
    if token:
        fd, askpass_path = tempfile.mkstemp(prefix="t0401-git-askpass-")
        with os.fdopen(fd, "w", encoding="utf-8") as stream:
            stream.write('#!/bin/sh\ncase "$1" in *Username*) echo x-access-token ;; *) echo "$T0401_GITHUB_TOKEN" ;; esac\n')
        os.chmod(askpass_path, 0o700)
        git_env.update({"GIT_ASKPASS": askpass_path, "GIT_TERMINAL_PROMPT": "0", "T0401_GITHUB_TOKEN": token})
    try:
        if REPO_DIR.exists():
            subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True, env=git_env)
            subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
            subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{BRANCH}"], check=True)
        else:
            subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)], check=True, env=git_env)
    finally:
        if askpass_path and os.path.exists(askpass_path):
            os.remove(askpass_path)
        git_env.pop("T0401_GITHUB_TOKEN", None)
        token = None

clone_or_update_branch()
os.chdir(PROJECT_DIR)
assert subprocess.check_output(["git", "branch", "--show-current"], text=True).strip() == BRANCH
print("Checked out project branch:", BRANCH)


In [ ]:
requirements = Path("requirements.txt").read_text(encoding="utf-8").splitlines()
filtered = [line for line in requirements if line.strip() and not line.lstrip().startswith("#")
            and line.split("=", 1)[0].strip().lower() not in {"torch", "torchvision"}]
if sys.version_info >= (3, 13):
    filtered = ["scikit-learn==1.6.1" if line.strip().lower().startswith("scikit-learn==") else line for line in filtered]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *filtered], check=True)
import torchvision
print("Retained runtime PyTorch/Torchvision:", torch.__version__, torchvision.__version__)

raw_csv = PROJECT_DIR / "data/raw/fer2013.csv"
drive_data = DRIVE_ROOT / "fer2013.csv"
raw_csv.parent.mkdir(parents=True, exist_ok=True)
if not raw_csv.is_file() and drive_data.is_file():
    shutil.copy2(drive_data, raw_csv)
if not raw_csv.is_file():
    token = None
    try:
        token = userdata.get("KAGGLE_API_TOKEN")
    except Exception:
        pass
    if not token:
        raise RuntimeError("Thi?u Colab Secret KAGGLE_API_TOKEN ho?c CSV trong Drive.")
    os.environ["KAGGLE_API_TOKEN"] = token
    token = None
    try:
        subprocess.run([sys.executable, "-m", "src.data.downloader"], cwd=PROJECT_DIR, check=True)
    finally:
        os.environ.pop("KAGGLE_API_TOKEN", None)
from src.data.downloader import FER2013Downloader
split_counts = FER2013Downloader.validate_csv(raw_csv)
assert split_counts == {"Training": 28709, "PublicTest": 3589, "PrivateTest": 3589}, split_counts
if not drive_data.exists():
    shutil.copy2(raw_csv, drive_data)

# Redirect generated artifacts to local SSD; sync to Drive after each run.
outputs = PROJECT_DIR / "outputs"
outputs.mkdir(exist_ok=True)
checkpoint_link = outputs / "checkpoints"
if checkpoint_link.is_symlink():
    if checkpoint_link.resolve() != LOCAL_CHECKPOINTS.resolve():
        raise RuntimeError("outputs/checkpoints symlink tr? sai; d?ng ?? b?o to?n d? li?u.")
elif checkpoint_link.exists():
    if any(item.name != ".gitkeep" for item in checkpoint_link.iterdir()):
        raise RuntimeError("outputs/checkpoints c? d? li?u; kh?ng ghi ??.")
    for item in checkpoint_link.iterdir():
        item.unlink()
    checkpoint_link.rmdir()
if not checkpoint_link.exists():
    checkpoint_link.symlink_to(LOCAL_CHECKPOINTS, target_is_directory=True)
print("Verified FER2013 splits:", split_counts)


In [ ]:
import numpy as np
import pandas as pd
import yaml
from mlflow.tracking import MlflowClient
from src.train.mlflow_logger import MLflowLogger

CONFIG_FILES = ["exp09_aug_none.yaml", "exp09_aug_basic.yaml", "exp10_aug_advanced.yaml"]
raw_configs = []
for filename in CONFIG_FILES:
    config_path = Path("experiments/ablation") / filename
    config = yaml.safe_load(config_path.read_text(encoding="utf-8"))
    config["config_path"] = str(config_path)
    config["run_name"] = Trainer._format_run_name(
        exp_id=config["exp_id"], architecture=config["architecture"],
        optimizer=config["optimizer"].lower(), pretrained=config["pretrained"],
        image_size=config["image_size"], augmentation_mode=config["augmentation_mode"])
    raw_configs.append(config)

def sanitize_to_python_scalars(record):
    def convert(value):
        if isinstance(value, np.generic): return convert(value.item())
        if isinstance(value, np.ndarray): return convert(value.item()) if value.ndim == 0 else str(convert(value.tolist()))
        if isinstance(value, dict): return str({str(key): convert(item) for key, item in value.items()})
        if isinstance(value, (list, tuple)): return str([convert(item) for item in value])
        if value is None or type(value) in (bool, int, float, str): return value
        return str(value)
    return {str(key): convert(value) for key, value in record.items()}

manifest = pd.DataFrame([sanitize_to_python_scalars(config) for config in raw_configs])
client = MlflowClient(tracking_uri=os.environ["MLFLOW_TRACKING_URI"])
experiment = client.get_experiment_by_name("BTL02_FER2013_PyTorch")
finished_names = set()
if experiment:
    for run in client.search_runs([experiment.experiment_id], max_results=10000):
        name = run.data.tags.get("mlflow.runName")
        if run.info.status == "FINISHED" and name:
            finished_names.add(name)
manifest["status"] = ["DONE" if row in finished_names else "PENDING" for row in manifest["run_name"]]
display(manifest[["exp_id", "architecture", "augmentation_mode", "seed", "batch_size", "epochs", "run_name", "status"]])


In [ ]:
LOCAL_CSV = Path("outputs/logs") / CSV_NAME
DRIVE_CSV.parent.mkdir(parents=True, exist_ok=True)
sync_lock = threading.Lock()


def sync_and_export_results(sync_checkpoints=True):
    with sync_lock:
        logger = MLflowLogger()
        raw_export = Path("outputs/logs/t0401_all_runs_export.csv")
        logger.export_summary_csv(str(raw_export))
        all_rows = pd.read_csv(raw_export, encoding="utf-8-sig", keep_default_na=False)
        assert all_rows.columns.tolist() == MLflowLogger.CSV_COLUMNS
        selected = all_rows[all_rows["Name"].astype(str).isin({item["run_name"] for item in raw_configs})].copy()
        selected = selected[selected["Status"].astype(str).str.upper().eq("FINISHED")]
        safe = MLflowLogger.normalize_records(selected.to_dict(orient="records"))
        result = pd.DataFrame(safe, columns=MLflowLogger.CSV_COLUMNS)
        temp_csv = LOCAL_CSV.with_name(LOCAL_CSV.name + ".sync-tmp")
        result.to_csv(temp_csv, index=False, encoding="utf-8-sig")
        os.replace(temp_csv, LOCAL_CSV)
        drive_temp = DRIVE_CSV.with_name(DRIVE_CSV.name + ".sync-tmp")
        shutil.copy2(LOCAL_CSV, drive_temp)
        os.replace(drive_temp, DRIVE_CSV)
        sync_tree_new_files(LOCAL_MLRUNS, DRIVE_MLRUNS, stable_age_seconds=3)
        if sync_checkpoints:
            sync_tree_new_files(LOCAL_CHECKPOINTS, DRIVE_CHECKPOINTS)
        raw_export.unlink(missing_ok=True)
        print(f"Synced {len(result)} FINISHED T0401 runs; CSV: {DRIVE_CSV}", flush=True)
        return result

print("CSV schema:", len(MLflowLogger.CSV_COLUMNS), "columns")


In [ ]:
failed_runs = []
for config in raw_configs:
    run_name = config["run_name"]
    experiment = client.get_experiment_by_name("BTL02_FER2013_PyTorch")
    matches = client.search_runs([experiment.experiment_id], filter_string=f"tags.mlflow.runName = '{run_name}'", max_results=100) if experiment else []
    if any(run.info.status == "FINISHED" for run in matches):
        print(f"[AUTO-SKIP] FINISHED: {run_name}", flush=True)
        sync_and_export_results()
        continue
    print(f"\n===== {config['exp_id']} / {config['augmentation_mode']} =====", flush=True)
    process = subprocess.Popen([sys.executable, "-u", "-m", "src.train.trainer", "--config", config["config_path"]],
                               cwd=PROJECT_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1, env=os.environ.copy())
    for line in process.stdout:
        print(line, end="", flush=True)
    code = process.wait()
    if code:
        failed_runs.append({"run_name": run_name, "return_code": code})
        print(f"[FAILED] exit={code}; l?u tr?ng th?i v? ti?p t?c run k? ti?p.", flush=True)
    else:
        print(f"[DONE] {run_name}", flush=True)
    sync_and_export_results(sync_checkpoints=(code == 0))
sync_and_export_results(sync_checkpoints=True)
if failed_runs:
    print("Runs c?n ?i?u tra/ch?y l?i:", failed_runs)

final_results = pd.read_csv(DRIVE_CSV, encoding="utf-8-sig", keep_default_na=False) if DRIVE_CSV.is_file() else pd.DataFrame(columns=MLflowLogger.CSV_COLUMNS)
assert final_results.columns.tolist() == MLflowLogger.CSV_COLUMNS
print("Completed rows:", len(final_results), "/ 3")
display(final_results[["Name", "Run ID", "Status", "params.augmentation", "metrics.best_epoch",
                       "metrics.train_loss", "metrics.train_acc", "metrics.best_val_acc",
                       "metrics.private_test_acc", "metrics.macro_f1"]])
print("Drive CSV:", DRIVE_CSV)
print("Drive checkpoints:", DRIVE_CHECKPOINTS)
print("Drive MLflow:", DRIVE_MLRUNS)
